In [7]:
import math

# Parameters (adjust as needed)
diagonal = 15.00+2*0.05*2/math.sqrt(3)+0.001  # Diagonal distance across the hexagon (mm)
apothem = (diagonal) / 2 * math.sqrt(3) / 2.0  # Apothem (distance from center to flat side)
side_length = 2*apothem  # Side length
a1_x = side_length  # Primitive vector 1 x-component
a1_y = 0.0  # Primitive vector 1 y-component
a2_x = side_length / 2.0
a2_y = side_length * math.sqrt(3.0) / 2.0

min_N = 13  # Start from this ring (note: example code has 13, but comment mentions ring 1; adjust if typo)
max_N = 13 + (3 - 1)  # End at this ring (note: example code ends at 15, but comment mentions ring 6; adjust if needed)

# Open file for writing coordinates (x y z in mm, without units)
with open("coordinates.txt", "w") as coord_file:
    for n1 in range(-max_N, max_N + 1):
        for n2 in range(max(-max_N, -n1 - max_N), min(max_N, -n1 + max_N) + 1):
            # Calculate the "ring" distance from origin using the max norm
            ring = max(abs(n1), abs(n2), abs(n1 + n2))
            if ring >= min_N and ring <= max_N:
                # Calculate position using primitive vectors
                x = n1 * a1_x + n2 * a2_x
                y = n1 * a1_y + n2 * a2_y
                z = 0.0  # Adjust if prisms are offset along z
                # Write to file
                coord_file.write(f"{x} {y} {z}\n")

In [13]:
r_start=15
thickness=2
crystal_xysize=2 #mm
crystal_zsize=10 #mm
thickness_z=2
points = set()


for i in range(r_start,r_start+thickness):
    r=i
    for z in range(0,thickness_z):
        x = 0
        y = r
        d = 1 - r
        while y >= x:
            if z==0:
                points.add((x, y,z))
                points.add((-x, y,z))
                points.add((x, -y,z))
                points.add((-x, -y,z))
                points.add((y, x,z))
                points.add((-y, x,z))
                points.add((y, -x,z))
                points.add((-y, -x,z))
            else:
                points.add((x, y,z))
                points.add((-x, y,z))
                points.add((x, -y,z))
                points.add((-x, -y,z))
                points.add((y, x,z))
                points.add((-y, x,z))
                points.add((y, -x,z))
                points.add((-y, -x,z))

                points.add((x, y,-z))
                points.add((-x, y,-z))
                points.add((x, -y,-z))
                points.add((-x, -y,-z))
                points.add((y, x,-z))
                points.add((-y, x,-z))
                points.add((y, -x,-z))
                points.add((-y, -x,-z))

            x += 1
            if d <= 0:
                d += 2 * x + 1
            else:
                y -= 1
                d += 2 * (x - y) + 1

with open("coordinates.txt", "w") as coord_file:
    for point in points:
        print(f"{point[0]*crystal_xysize} {point[1]*crystal_xysize} {point[2]*crystal_zsize/2}", file=coord_file)

In [55]:
crystal_xysize = 4+2*0.05#+0.01  # mm
r_start = int(192/crystal_xysize)
thickness =int(35/crystal_xysize)
crystal_zsize = 20+2*0.05  # mm
thickness_z = 8

points = set()

# Calculate the inner and outer radius bounds
r_inner = r_start
r_outer = r_start + thickness-1

# Iterate over a bounding box large enough to cover the outer radius
max_r = r_outer + 1  # Add a bit of margin for discrete points
for x in range(-max_r, max_r + 1):
    for y in range(-max_r, max_r + 1):
        if x == 0 and y == 0:
            continue  # Skip center if not wanted
        r = (x**2 + y**2)**0.5
        if r_inner <= r < r_outer:
            # Add points for each z layer
            for z in range(-thickness_z + 1, thickness_z):  # Symmetric around 0
                points.add((x, y, z))

# Note: This assumes you want points at integer z from -(thickness_z-1) to (thickness_z-1),
# but adjust based on exact z requirements. Original had z=0 and z=±1 for thickness_z=2.

with open("coordinates.txt", "w") as coord_file:
    for point in sorted(points):  # Sort for consistent order
        scaled_x = point[0] * crystal_xysize
        scaled_y = point[1] * crystal_xysize
        scaled_z = point[2] * (crystal_zsize)  # Original scaling for z
        print(f"{scaled_x} {scaled_y} {scaled_z}", file=coord_file)

In [22]:
r_start

91

In [2]:
int(192/(4+2*0.05))*4

184

# Random Geometry

In [29]:
crystal_short = 4 + 2 * 0.05  # mm, for the square sides
crystal_long = 20 + 2 * 0.05  # mm, for the long side
half_short = crystal_short / 2
half_long = crystal_long / 2

n = 1  # Parameter: number of crystal stacking (layers in thickness)
k = 0.0  # Parameter: distance from center to inner face (mm)

m = int(round(n * crystal_long / crystal_short))  # Number of crystals along each in-plane direction to approximate cube

offset = - (m - 1) / 2.0 * crystal_short

walls = [
    ('x', 1), ('x', -1),
    ('y', 1), ('y', -1),
    ('z', 1), ('z', -1)
]

points = set()

for normal_dir, sign in walls:
    inner_center = sign * (k + half_long)
    if normal_dir == 'x':
        in1, in2 = 'y', 'z'
    elif normal_dir == 'y':
        in1, in2 = 'x', 'z'
    elif normal_dir == 'z':
        in1, in2 = 'x', 'y'
    
    for i in range(n):
        normal_pos = inner_center + i * sign * crystal_long
        for j1 in range(m):
            pos1 = offset + j1 * crystal_short
            for j2 in range(m):
                pos2 = offset + j2 * crystal_short
                x = 0.0
                y = 0.0
                z = 0.0
                if normal_dir == 'x':
                    x = normal_pos
                    y = pos1
                    z = pos2
                elif normal_dir == 'y':
                    y = normal_pos
                    x = pos1
                    z = pos2
                elif normal_dir == 'z':
                    z = normal_pos
                    x = pos1
                    y = pos2
                # Round to avoid floating point duplicates
                rx = round(x, 6)
                ry = round(y, 6)
                rz = round(z, 6)
                points.add((rx, ry, rz))

with open("coordinates.txt", "w") as coord_file:
    for point in sorted(points):
        print(f"{point[0]} {point[1]} {point[2]}", file=coord_file)

In [28]:

# Example: Generate for Design 1 or 2 (cube)
generate_cube_coords(
    pitch_xy=4.1,          # Pitch in xy (mm)
    pitch_z=20.1,          # Pitch in z (mm)
    r_start_mm=10.0,      # Inner half-side (mm)
    thickness_mm=5.0,     # Thickness (mm) - smaller for minimal crystals in Design 1, larger for denser Design 2
    with_rotations=True,   # Include rotations (True for 5 params per line, False for 3)
    output='coordinates.txt'  # Output file
)

## Example: Generate for Design 3 (cylinder)
#generate_cylinder_coords(
#    pitch_xy=4.1,
#    pitch_z=20.1,
#    r_start_mm=192.0,      # Inner radius (mm)
#    thickness_mm=35.0,     # Radial thickness (mm)
#    z_layers=8,            # Half-number of z layers (e.g., 8 for -7 to 7)
#    with_rotations=True,
#    output='cylinder_coords.txt'
#)
#
## Example: Generate for Design 4 or 5 (sphere)
#generate_sphere_coords(
#    pitch_xy=4.1,
#    pitch_z=20.1,          # Set equal to pitch_xy for more isotropic sphere in Design 5
#    r_start_mm=192.0,      # Inner radius (mm)
#    thickness_mm=35.0,     # Thickness (mm) - larger for denser Design 4
#    with_rotations=True,
#    output='sphere_coords.txt'
#)

# After running, check the output .txt files for the coordinates.
# Load into Geant4 as before.